# Tuning v3 — Pseudo-absence ratio sensitivity analysis

## What is a sensitivity analysis?
It means testing whether a design decision made "by hand" (here: how many
pseudo-absences to sample per observed fire) is really the best choice, or
whether the model's outcome changes little/a lot when it is varied. Instead of
assuming the 2:1 ratio used so far is correct, several reasonable alternatives
are tested and compared under the same validation protocol.

## Why test several ratios?
Fire is a rare event (~2.5% of pixel-years). To train a classifier, a subset of
"unburned" points (pseudo-absences) is sampled instead of using every fire-free
pixel — otherwise the model would learn to always say "no burn" and would be
right 97.5% of the time without being useful (see `WORKFLOW.md`).
The pseudo-absence:presence ratio controls that balance:
- Lower ratios (1:1) → more balanced classes, but the model sees less
  variety of "unburned" conditions and may over-predict susceptibility.
- Higher ratios (3:1) → more variety of "unburned" conditions, but the classes
  become imbalanced again, moving back toward the original problem.
There is no way to know which is best without testing it empirically — hence
this sensitivity analysis.

## How is the best ratio selected?
For each candidate ratio (1:1, 2:1, 3:1), **exactly the same
`tuning/v2` pipeline** is repeated: `GridSearchCV` (10-fold spatial-block CV, `scoring='average_precision'`)
restricted to `year<=2019` (no temporal leakage), using the same hyperparameter
grids as v1/v2, followed by full evaluation (10-fold spatial block CV +
temporal hold-out `>=2020`, with confusion matrix). The winning ratio is
the one that achieves the best **mean spatial PR-AUC** on Random Forest — the
selection metric used throughout the project, and the model with the most
consistent performance in v1 and v2. The Logistic Regression result is also
reported to check whether both models agree on the optimal ratio.

## Improvement over v1 and v2
- **v1** tuned hyperparameters but left a data design decision (the
  2:1 ratio) fixed without questioning it, and also had temporal leakage in the tuning.
- **v2** fixed the temporal leakage, but still assumed the 2:1 ratio was
  correct — it was never tested whether another ratio gave better performance.
- **v3** questions that assumption: it keeps v2's fix (tuning only with
  `year<=2019`) and adds a new search dimension (the sampling ratio),
  identifying the best sample size instead of assuming it.

## What this notebook does
1. Rebuilds the base dataset (`pixel_year_full.csv`, already generated with Earth
   Engine — EE is not queried again here) and applies the same 3 km exclusion
   buffer used originally.
2. For each ratio (1:1, 2:1, 3:1): samples the dataset, runs `GridSearchCV`
   (`year<=2019`) for LR and RF, and evaluates with v2's full protocol.
3. Compares the 3 ratios and picks the best one by RF's spatial PR-AUC.
4. Saves all results in this folder (`tuning/v3/`):
   `tuning_v3_sensitivity_metrics.csv` (all 3 ratios) and
   `tuning_v3_final_metrics.csv` (only the winning ratio).

In [1]:
# === Tuning v3 — Setup: base data for the sensitivity analysis ===
# We reuse the FULL pixel-year table (before any stratified sampling),
# generated once with Earth Engine in
# model/logistic_regression/baseline_comparison.ipynb. This avoids querying
# Earth Engine again: we only change HOW MANY pseudo-absences are sampled per
# presence (the "ratio"), reusing the same stratified sampling logic + 3 km
# exclusion buffer that generated model_dataset.csv (ratio 2:1).
import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              confusion_matrix)
from sklearn.base import clone

pixel_year = pd.read_csv('../../data/model_dataset/pixel_year_full.csv')
pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']
BLOCK = 0.25
EXCL_BUFFER = 3000            # 3 km — identical to the one used to build model_dataset.csv (ratio 2:1)
buffer_deg = EXCL_BUFFER / 111000

print("pixel_year_full:", pixel_year.shape,
      "| presences (burned=1):", int((pixel_year['burned']==1).sum()),
      "| candidate absences (burned=0):", int((pixel_year['burned']==0).sum()))

# --- Eligible presences and absences ---
# The 3 km exclusion buffer does NOT depend on the ratio (it only depends on where
# the presences are), so it is computed ONCE and reused for the 3 scenarios.
presences = pixel_year[pixel_year['burned'] == 1].copy()
absences_all = pixel_year[pixel_year['burned'] == 0].copy()

kept_absences = []
for y in sorted(pixel_year['year'].unique()):
    pres_y = presences[presences['year'] == y][['lon','lat']].values
    abs_y  = absences_all[absences_all['year'] == y]
    if len(pres_y) == 0:
        kept_absences.append(abs_y); continue
    tree = cKDTree(pres_y)
    dists, _ = tree.query(abs_y[['lon','lat']].values, k=1)
    kept_absences.append(abs_y[dists > buffer_deg])
absences_far = pd.concat(kept_absences, ignore_index=True)
print(f"Absences after 3 km buffer: {len(absences_far)} (of {len(absences_all)} candidates)")
print(f"Max ratio usable: 1:{len(absences_far)/len(presences):.1f}")

pixel_year_full: (83184, 13) | presences (burned=1): 2077 | candidate absences (burned=0): 81107
Absences after 3 km buffer: 78878 (of 81107 candidates)
Max ratio usable: 1:38.0


In [2]:
def build_ratio_dataset(ratio, seed=42):
    """
    Builds a modelling dataset with ALL presences (observed fires)
    and `ratio` times that many pseudo-absences, randomly sampled
    from the eligible absences (already filtered by the 3 km exclusion
    buffer). This is exactly the same logic that generated
    data/model_dataset/model_dataset.csv (where ratio=2), but reusable for
    any ratio.
    """
    n_abs = min(len(absences_far), int(round(ratio * len(presences))))
    absences_sample = absences_far.sample(n=n_abs, random_state=seed)
    df = pd.concat([presences, absences_sample], ignore_index=True) \
           .sample(frac=1, random_state=seed).reset_index(drop=True)
    df['block'] = (df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                  (df['lat']//BLOCK).astype(int).astype(str)
    return df


def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=BLOCK):
    """Validation protocol IDENTICAL to tuning/v2 (10-fold spatial block CV
    over the full dataset + temporal hold-out train<=2019/test>=2020), with
    confusion matrices. Reused unchanged so that v2 and v3 are
    directly comparable."""
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values
    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    cm_spatial = np.zeros((2, 2), dtype=int)
    for tr, te in gkf.split(X, y, groups):
        m = clone(estimator).fit(X[tr], y[tr])
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1 = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        cm_spatial += confusion_matrix(y[te], pred, labels=[0, 1])
    r = np.array(rows)
    print(f"  [{name}] SPATIAL  AUC={r[:,0].mean():.3f}±{r[:,0].std():.3f}  "
          f"PR-AUC={r[:,1].mean():.3f}±{r[:,1].std():.3f}  F1={r[:,2].mean():.3f}±{r[:,2].std():.3f}")

    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])
    cm_temporal = confusion_matrix(y[te], pred, labels=[0, 1])
    auc_t = roc_auc_score(y[te], prob)
    prauc_t = average_precision_score(y[te], prob)
    f1_t = f1_score(y[te], pred)
    print(f"  [{name}] TEMPORAL AUC={auc_t:.3f}  PR-AUC={prauc_t:.3f}  F1={f1_t:.3f}")

    return {
        'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0), 'spatial_cm': cm_spatial,
        'temporal': (auc_t, prauc_t, f1_t), 'temporal_cm': cm_temporal,
    }


def cm_to_dict(cm, prefix):
    tn, fp, fn, tp = cm.ravel()
    return {f'{prefix}_tn': int(tn), f'{prefix}_fp': int(fp),
            f'{prefix}_fn': int(fn), f'{prefix}_tp': int(tp)}

In [3]:
def run_ratio_pipeline(ratio):
    """
    Complete pipeline for ONE pseudo-absence ratio, identical to
    tuning/v2 except for the input dataset:
      1. Builds the dataset with that ratio (all presences + ratio x absences).
      2. GridSearchCV (10-fold spatial-block CV, PR-AUC) ONLY with year<=2019 for
         LR and RF, with the SAME grids as v1/v2 (to isolate the effect of
         changing the ratio, without changing anything else).
      3. Evaluates both tuned models with the full protocol (spatial CV +
         temporal) on the COMPLETE dataset for that ratio (all years).
    Returns a list of 2 rows (LR, RF) ready for the comparison DataFrame.
    """
    print(f"\n{'='*70}\nRATIO {ratio}:1 (pseudo-absences : presences)\n{'='*70}")
    ratio_df = build_ratio_dataset(ratio)
    n_pres = int(ratio_df['burned'].sum())
    print(f"Dataset ratio {ratio}:1 -> {ratio_df.shape[0]} rows "
          f"({n_pres} presences, {ratio_df.shape[0]-n_pres} pseudo-absences)")

    # ---- Tuning: ONLY year<=2019 (same temporal-leakage fix as v2) ----
    tune_df = ratio_df[ratio_df['year'] <= 2019].copy()
    X_tune = tune_df[pred_cols].values
    y_tune = tune_df['burned'].astype(int).values
    groups_tune = tune_df['block'].values
    cv = GroupKFold(n_splits=10)

    pipeline_lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=5000))])
    param_grid_lr = {'clf__C': [0.01, 0.1, 1.0, 10.0], 'clf__penalty': ['l1', 'l2'], 'clf__solver': ['liblinear']}
    lr_search = GridSearchCV(pipeline_lr, param_grid_lr, cv=cv, scoring='average_precision', n_jobs=-1)
    lr_search.fit(X_tune, y_tune, groups=groups_tune)

    param_grid_rf = {'n_estimators': [200, 300, 500], 'max_features': ['sqrt', 0.5],
                      'min_samples_leaf': [3, 5, 10, 20], 'max_depth': [None, 10, 20]}
    rf_search = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid_rf,
                              cv=cv, scoring='average_precision', n_jobs=-1)
    rf_search.fit(X_tune, y_tune, groups=groups_tune)

    print("Best LR:", lr_search.best_params_)
    print("Best RF:", rf_search.best_params_)

    # ---- Final evaluation: COMPLETE dataset for this ratio (all years) ----
    res_lr = evaluate_model(f"LR ratio {ratio}:1", lr_search.best_estimator_, ratio_df, pred_cols)
    res_rf = evaluate_model(f"RF ratio {ratio}:1", rf_search.best_estimator_, ratio_df, pred_cols)

    out = []
    for model_name, res, best_params in [
        ('Logistic Regression', res_lr, lr_search.best_params_),
        ('Random Forest', res_rf, rf_search.best_params_),
    ]:
        row = {
            'ratio': f'{ratio}:1', 'model': model_name, 'best_params': str(best_params),
            'n_rows': ratio_df.shape[0], 'n_presences': n_pres,
            'auc_spatial_mean': res['spatial'][0], 'auc_spatial_std': res['spatial_std'][0],
            'prauc_spatial_mean': res['spatial'][1], 'prauc_spatial_std': res['spatial_std'][1],
            'f1_spatial_mean': res['spatial'][2], 'f1_spatial_std': res['spatial_std'][2],
            'auc_temporal': res['temporal'][0], 'prauc_temporal': res['temporal'][1],
            'f1_temporal': res['temporal'][2],
        }
        row.update(cm_to_dict(res['spatial_cm'], 'cm_spatial'))
        row.update(cm_to_dict(res['temporal_cm'], 'cm_temporal'))
        out.append(row)
    return out

In [4]:
# Ratio 1:1 (balanced classes)
results_list = []
results_list += run_ratio_pipeline(1)


RATIO 1:1 (pseudo-ausencias : presencias)
Dataset ratio 1:1 -> 4154 filas (2077 presencias, 2077 pseudo-ausencias)


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Best LR: {'clf__C': 0.01, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Best RF: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 200}


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_

  [LR ratio 1:1] SPATIAL  AUC=0.846±0.032  PR-AUC=0.823±0.060  F1=0.754±0.079
  [LR ratio 1:1] TEMPORAL AUC=0.810  PR-AUC=0.747  F1=0.663
  [RF ratio 1:1] SPATIAL  AUC=0.878±0.027  PR-AUC=0.866±0.036  F1=0.790±0.047
  [RF ratio 1:1] TEMPORAL AUC=0.817  PR-AUC=0.704  F1=0.684


In [5]:
# Ratio 2:1 (the one used so far in v1/v2 -- also serves as a consistency
# check: it should reproduce v2's numbers almost exactly, since it uses
# the same seed (42) and the same number of pseudo-absences)
results_list += run_ratio_pipeline(2)


RATIO 2:1 (pseudo-ausencias : presencias)
Dataset ratio 2:1 -> 6231 filas (2077 presencias, 4154 pseudo-ausencias)


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best LR: {'clf__C': 0.01, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Best RF: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 300}


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ra

  [LR ratio 2:1] SPATIAL  AUC=0.832±0.064  PR-AUC=0.703±0.105  F1=0.613±0.123
  [LR ratio 2:1] TEMPORAL AUC=0.808  PR-AUC=0.617  F1=0.557


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  [RF ratio 2:1] SPATIAL  AUC=0.877±0.041  PR-AUC=0.779±0.067  F1=0.680±0.074
  [RF ratio 2:1] TEMPORAL AUC=0.819  PR-AUC=0.569  F1=0.560


In [6]:
# Ratio 3:1 (more pseudo-absences -- classes imbalanced again)
results_list += run_ratio_pipeline(3)


RATIO 3:1 (pseudo-ausencias : presencias)
Dataset ratio 3:1 -> 8308 filas (2077 presencias, 6231 pseudo-ausencias)


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best LR: {'clf__C': 0.01, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Best RF: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 300}


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ra

  [LR ratio 3:1] SPATIAL  AUC=0.831±0.051  PR-AUC=0.618±0.113  F1=0.504±0.167
  [LR ratio 3:1] TEMPORAL AUC=0.806  PR-AUC=0.542  F1=0.486
  [RF ratio 3:1] SPATIAL  AUC=0.873±0.032  PR-AUC=0.702±0.067  F1=0.603±0.098
  [RF ratio 3:1] TEMPORAL AUC=0.817  PR-AUC=0.492  F1=0.493


In [7]:
# === Save the full sensitivity-analysis results ===
# One row per combination (ratio x model) = 3 ratios x 2 models = 6 rows, with
# the same columns as tuning_v2_metrics.csv plus 'ratio', 'n_rows' and
# 'n_presences', to allow comparing ratios at a glance.
results_v3_df = pd.DataFrame(results_list)
out_path = 'tuning_v3_sensitivity_metrics.csv'   # saved in this same folder (tuning/v3)
results_v3_df.to_csv(out_path, index=False)
print(f"Results saved to: tuning/v3/{out_path}\n")
results_v3_df[['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
               'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']]

Resultados guardados en: tuning/v3/tuning_v3_sensitivity_metrics.csv



,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,1:1,Logistic Regression,4154,2077,0.845518,0.822546,0.753945,0.809537,0.746931,0.663230
1,1:1,Random Forest,4154,2077,0.878350,0.865608,0.789538,0.817119,0.703800,0.684303
2,2:1,Logistic Regression,6231,2077,0.832055,0.702716,0.613300,0.808193,0.616809,0.557166
3,2:1,Random Forest,6231,2077,0.876874,0.779414,0.680450,0.819263,0.568723,0.560000
4,3:1,Logistic Regression,8308,2077,0.831386,0.618466,0.503994,0.806410,0.541892,0.486486
5,3:1,Random Forest,8308,2077,0.873274,0.702023,0.602735,0.817056,0.492481,0.492936


In [8]:
# === Selecting the best ratio + final model ===
# Criterion: best mean spatial PR-AUC on Random Forest (the same selection
# metric used throughout the project, and the model with the most consistent
# performance in v1 and v2). The Logistic Regression ranking is also reported
# to check whether both models agree on the optimal ratio.
rf_rows = results_v3_df[results_v3_df['model'] == 'Random Forest'].sort_values('prauc_spatial_mean', ascending=False)
lr_rows = results_v3_df[results_v3_df['model'] == 'Logistic Regression'].sort_values('prauc_spatial_mean', ascending=False)

best_ratio_rf = rf_rows.iloc[0]['ratio']
best_ratio_lr = lr_rows.iloc[0]['ratio']

print("Random Forest ranking by spatial PR-AUC:")
print(rf_rows[['ratio','prauc_spatial_mean','prauc_temporal']].to_string(index=False))
print("\nLogistic Regression ranking by spatial PR-AUC:")
print(lr_rows[['ratio','prauc_spatial_mean','prauc_temporal']].to_string(index=False))

print(f"\n>>> Best ratio for Random Forest: {best_ratio_rf}")
print(f">>> Best ratio for Logistic Regression: {best_ratio_lr}")

# The final model has already been trained and evaluated as part of the analysis
# above (no need to retrain: same data + same seed = same result).
# It is extracted and saved separately so there is an easy-to-cite "official"
# file, in addition to being included in tuning_v3_sensitivity_metrics.csv.
final_df = results_v3_df[results_v3_df['ratio'] == best_ratio_rf].copy()
final_out_path = 'tuning_v3_final_metrics.csv'
final_df.to_csv(final_out_path, index=False)
print(f"\nFinal model (optimal ratio = {best_ratio_rf}, chosen by RF spatial PR-AUC) "
      f"saved to: tuning/v3/{final_out_path}")
final_df[['ratio','model','best_params','auc_spatial_mean','prauc_spatial_mean','f1_spatial_mean',
          'auc_temporal','prauc_temporal','f1_temporal']]

Ranking Random Forest por PR-AUC espacial:
ratio  prauc_spatial_mean  prauc_temporal
  1:1            0.865608        0.703800
  2:1            0.779414        0.568723
  3:1            0.702023        0.492481

Ranking Logistic Regression por PR-AUC espacial:
ratio  prauc_spatial_mean  prauc_temporal
  1:1            0.822546        0.746931
  2:1            0.702716        0.616809
  3:1            0.618466        0.541892

>>> Mejor ratio para Random Forest: 1:1
>>> Mejor ratio para Logistic Regression: 1:1

Modelo final (ratio optimo = 1:1, elegido por PR-AUC espacial de RF) guardado en: tuning/v3/tuning_v3_final_metrics.csv


,ratio,model,best_params,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,1:1,Logistic Regression,"{'clf__C': 0.01, 'clf__penalty': 'l2', 'clf__s...",0.845518,0.822546,0.753945,0.809537,0.746931,0.663230
1,1:1,Random Forest,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.878350,0.865608,0.789538,0.817119,0.703800,0.684303
